In [10]:
import uproot
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import numpy as np
import os
import glob
import awkward as ak
import boost_histogram as bh
import mplhep as mh
from matplotlib.colors import LogNorm
import math
mh.style.use("ATLAS")
import particle
# !pip install pyarrow
# !pip install "pyarrow<21" --break-system-packages

In [39]:
def print_cutflow(filpaths, lumi_sample=10_000, lumi_target=None):
    if isinstance(filpaths, str):
        filpaths = [filpaths]

    df = uproot.concatenate([f"{filpath}:cutflow_by_category/CC_nu_mu_fiducial" for filpath in filpaths], library="pd")
    # df = uproot.concatenate([f"{filpath}:cutflow" for filpath in filpaths], library="pd")
    # cutflow tables from each run are appended together in the root file, so we need to merge all entries with the same cut_name. We can do this by grouping by cut_name and summing the other columns.
    df = df.groupby("cut_name").sum().reset_index()
    df = df.sort_values("all", ascending=False)

    # Need to recompute the efficiency column, as it is not meaningful after summing the other columns.
    df["efficiency"] = df["passed"] / df["all"]

    if lumi_target is not None:
        df["all"] = df["all"] * (lumi_target / lumi_sample)
        df["passed"] = df["passed"] * (lumi_target / lumi_sample)

    display(df)
    
    

In [43]:
print_cutflow("build/2022_2023.bck.root", lumi_sample=10_000, lumi_target=66.69)

,cut_name,all,passed,efficiency
0,At least one long track,1096.897113,610.686999,0.556740
5,Remove CaloNu PMT region (truth),1096.897113,1096.897113,1.000000
12,Veto20 and Veto21 charge > 40 pC,610.686999,610.326873,0.999410
6,Timing Station Charge > 20 pC,610.326873,610.173486,0.999749
4,Preshower Charge > 2.5 pC,610.173486,604.118034,0.990076
10,Track pz > 50 GeV,604.118034,572.280228,0.947299
1,Leading track has at >= 7 layers,572.280228,564.010668,0.985550
9,Track nDoF >= 9,564.010668,541.822905,0.960661
2,Leading track has chi2/ndof < 15,541.822905,541.589490,0.999569
8,Track R at max radius < 95 mm,541.589490,369.429255,0.682120


In [44]:
print_cutflow("build/2024_noCaloNu.bck.root", lumi_sample=10_000, lumi_target=50.10)

,cut_name,all,passed,efficiency
0,At least one long track,802.61703,445.36395,0.554890
5,Remove CaloNu PMT region (truth),802.61703,802.61703,1.000000
12,Veto20 and Veto21 charge > 40 pC,445.36395,445.15353,0.999528
6,Timing Station Charge > 20 pC,445.15353,445.05333,0.999775
4,Preshower Charge > 2.5 pC,445.05333,440.58942,0.989970
10,Track pz > 50 GeV,440.58942,416.36106,0.945009
1,Leading track has at >= 7 layers,416.36106,410.28894,0.985416
9,Track nDoF >= 9,410.28894,393.84111,0.959912
2,Leading track has chi2/ndof < 15,393.84111,393.65574,0.999529
8,Track R at max radius < 95 mm,393.65574,268.38069,0.681765


In [45]:
print_cutflow("build/2024_CaloNu.bck.root", lumi_sample=10_000, lumi_target=70.03)

,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),636.530682,630.403057,0.990373
0,At least one long track,630.403057,333.959064,0.529755
12,Veto20 and Veto21 charge > 40 pC,333.959064,333.734968,0.999329
6,Timing Station Charge > 20 pC,333.734968,333.664938,0.999790
4,Preshower Charge > 2.5 pC,333.664938,330.093408,0.989296
10,Track pz > 50 GeV,330.093408,304.623497,0.922840
1,Leading track has at >= 7 layers,304.623497,297.634503,0.977057
9,Track nDoF >= 9,297.634503,280.778282,0.943366
2,Leading track has chi2/ndof < 15,280.778282,280.666234,0.999601
8,Track R at max radius < 95 mm,280.666234,196.847327,0.701357
